# ICC132 -Actividad integradora: implementación de un esquema asimétrico de firma y verificación

Esta actividad tiene como propósito realizar la implementación de un criptosistema asimétrico funconal. El esquema asimétrico elegido será: **RSA**

## Generación de llaves

A continuación, se generan un par de llaves:
* Llave pública
* Llave privada

In [93]:
import random
import hashlib

In [94]:
def es_primo(n):
    if n <= 1:
        return False
    for i in range(2, int(n**0.5) + 1):
        if n % i == 0:
            return False
    return True

In [95]:
def generar_primo(minimo, maximo):
    while True:
        n = random.randint(minimo, maximo)
        if es_primo(n):
            return n

In [96]:
p = generar_primo(100, 500)
q = generar_primo(100, 500)
while q == p:
    q = generar_primo(100, 500)

In [97]:
print(f"p: {p}, q: {q}")

p: 269, q: 167


`p` y `q` representan números primos, los cuales deben ser grandes, para esta práctica no trabajaremos con números robustos. Además, se utlizó `random.randint()` para mantener sencillez en la implementación. Por el momento, no experimentaremos con funciones como `os.urandom()`, aunque no se descarta la idea de que garantiza mayor seguridad al incluir bytes totalmente aleatorios directamemente del hardware.

In [98]:
n = p * q
phi = (p - 1) * (q - 1)

In [99]:
print(f"n: {n}, phi: {phi}")

n: 44923, phi: 44488


* `n` se calcula con `p * q`. Es el módulo base con el cual se realizarán todas las operaciones del sistema. Aparece en ambas llaves tanto pública como privada.
* $\phi(n)$ representa cuantos numeros entre 1 y `n` no comparten factores con `n`. Entonces, decide el espacio donde se calcularán las llaves.

`phi` en todo momento debe ser privado, si alguien conoce `phi` puede calcular `d` (llave privada de firma). Por lo tanto, `p` y `q` también deben mantenerse secretas ya que de ellos deriva `phi`.

In [100]:
def mcd(a, b):
    while b > 0:
        a, b = b, a % b
    return a

In [101]:
def generar_e(phi):
    e = random.randint(2, phi)
    while mcd(e, phi) != 1:
        e = random.randint(2, phi)
    return e

In [102]:
e = generar_e(phi)
print(f"e: {e}")
print(f"mcd(e, phi): {mcd(e, phi)}")

e: 36677
mcd(e, phi): 1


`e` es el exponente público usado para verificar la firma. Es público, por lo tanto forma parte de la llave pública junto con `n`. Además, `e` siempre deber se coprimo con `phi`. Si no son coprimos no existe el inverso multiplicativo, entonces no se puede calcular `d`. 

In [103]:
def algoritmo_euclides_extendido(a, b):
    if a == 0:
        return b, 0, 1
    mcd, x1, y1 = algoritmo_euclides_extendido(b % a, a)
    x = y1 - (b // a) * x1
    y = x1
    return mcd, x, y

def inverso_multiplicativo(e, phi):
    mcd, x, _ = algoritmo_euclides_extendido(e, phi)
    if mcd != 1:
        raise ValueError("No existe inverso multiplicativo")
    return x % phi

In [104]:
d = inverso_multiplicativo(e, phi)
print(f"d: {d}")
print(f"Verificación: (e * d) % phi: {(e * d) % phi}")

d: 13077
Verificación: (e * d) % phi: 1


`d` es la llave privada. Aquí, se implementan dos conceptos, el inverso multiplicativo el cual se calcula con base al algoritmo de euclides extendido. Depende de `e` y `phi`por que `d` es el inverso multiplicativo de `e` en el espacio definido por `phi`. Sin conocer `e` y `phi`, no se puede calcular `d`. 

Por otro lado, la llave privada siempre deberá mantenerse en secreto, ya que si alguien obtiene `d` puede firmar mensajes haciendose pasar por otra persona. Por lo que cualquier firma parecerá legítima porque se verificará correctamente con la llave pública.

En resumen, tenemos lo siguiente: 

* `n` = público
* `e` = público
* `d` = privado
* `p` = privado
* `q` = privado
* `phi` = privado

- Llave pública: (e, n)
- Llave privada: (d, n)

## Preparación del mensaje

El mensaje no debe ser firmado directamente, se debe firmar el hash del mensaje. Esto se debe a que, si firmamos el mensaje directamente la integridad de la información corre riesgo, ya que si alguien lo modifica, el receptor jamás sabrá que manipularon el mensaje. En cambio, si firmamos el hash del mensaje, si el mensaje cambia aunque sea un solo carácter, el hash cambia completamente. Entonces cuando verificas, si es hash no coincide, sabes con certeza que el mensaje fue alterado.

In [105]:
def preparar_mensaje(mensaje, n):
    if mensaje is None or mensaje.strip() == "":
        raise ValueError("El mensaje no puede estar vacío")
    
    mensaje_bytes = mensaje.encode('utf-8')

    hash_mensaje = hashlib.sha256(mensaje_bytes)

    return int(hash_mensaje.hexdigest(), 16) % n

Para la preparación del mensaje nos aseguramos que el mensaje no sea nulo ni esté vacío, son casos que limitaremos ya que no tienen mucho sentido que existan. Luego, transformamos en el mensaje a bytes para aplicar `sha256`. Finalmente, convertimos el resultado a un número entero. 

NOTA: Intentamos correr el codigo con numeros primos grandes pero el codigo se volvio demasiaado lento. Se redujo el hash con módulo para que siempre sea menor que `n` y la verificación de la firma pueda realizarse correctamente.

In [106]:
print(preparar_mensaje("Hola mundo", n))

2710


# Operación principal del sistema

Para esta sección se aplicarán dos funciones: 
* `firmar`: Recibe el mensaje, le aplica la función de preparación para obtener el hash, y luego aplica `pow(hash, d, n)`. Lo que genera un numero entero que es la firma.
* `verificar`: Recibe el mensaje original y la firma. Calcula el hash del mensaje recibido con la función de preparación, después descifra la firma para recuperar el hash original y se comparan ambos hashes.

In [107]:
def firmar(mensaje, d, n):
    hash_mensaje = preparar_mensaje(mensaje, n)
    firma = pow(hash_mensaje, d, n)
    return firma

def verificar(mensaje, firma, e, n):
    hash_mensaje = preparar_mensaje(mensaje, n)
    hash_verificacion = pow(firma, e, n)
    return hash_mensaje == hash_verificacion

In [108]:
mensaje = "Hola mundo"
firma = firmar(mensaje, d, n)
print(f"Firma: {firma}")
print(f"Verificación: {verificar(mensaje, firma, e, n)}")

mensaje_alterado = "Hola Mundo" # cambiar M
print(f"Mensaje alterado (Verificación): {verificar(mensaje_alterado, firma, e, n)}")

Firma: 2285
Verificación: True
Mensaje alterado (Verificación): False


Como se puede observar cualquier cambio mínimo en el mensaje original hace que la verificación falle, esto principalmente por el algoritmo `sha256` el cual nos garantiza integridad en todo momento.